<a href="https://colab.research.google.com/github/ckrickyh/pythonTools/blob/main/airTableF1_uploadPhotos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
!pip install pyairtable
!apt-get install -qq xattr

In [82]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [83]:
import sys
sys.path.append('/content/drive/MyDrive/pyConfig')
from airtableconfig import tokenConfig, baseIdConfig

# Setting

In [84]:
# google folder

project = 'airTable'
fld = 'airTablePhotoUpload'
drive_path = f'/content/drive/MyDrive/Muni/{project}/{fld}'

In [ ]:
token = tokenConfig
baseId = baseIdConfig
tableName = 'tblico0z9dlmsf4Mr'

'''
baseId = 'apphtjuFujdfd647NaV7'
tableName = 'tblOhXdfdqpaDbX5LeKR'
# https://airtable.com/apphtjuFujdfd647NaV7/tblOhXdfdqpaDbX5LeKR/viw62XcVen1JVp2a7?blocks=hide
'''

In [86]:
from pyairtable import Api
import requests
import os
import pandas as pd
#from google.colab import drive
from pathlib import Path
import time
#google
#drive.mount('/content/drive')

# Retrieve google photo url

In [87]:
from subprocess import getoutput
from IPython.display import HTML


In [88]:
#注意路徑名
# %cd /content/gdrive/MyDrive/ToHyd/SWTCoverPhoto/
os.chdir(drive_path)

import pathlib
import pandas as pd

#df1=pd.DataFrame(columns = ["Path", "ShareLink"])

def get_shareable_link(file_path):
    fid = getoutput("xattr -p 'user.drive.id' " + "'" + file_path + "'")
    print(fid)
    # make a link and display it
    glink = HTML(f"<a href=https://drive.google.com/file/d/{fid} target=_blank>file URL</a>")
    return glink, fid


#files = !ls /content/gdrive/MyDrive/ImageAI/*.jpg
#for file_path in files:
    #print(file_path)

#注意路徑名
path = pathlib.Path(drive_path)


file_paths = []
links = []
fids = []
fileNames = []
fileFullNames = []
treeNos = []
for files in path.glob("**/*.*"):

  ###for files in path.glob("**/*.jpg"):        #all subfolder with .jpg
  print(files)

  #===================================== ### 檔案類型
  fileType = [".JPG", ".JPEG", ".PNG"]

  if str(files).upper().endswith(tuple(fileType)):

    file_path = str(files)   #for document
    print(file_path)
    # file_path = str(files.parent.absolute())    #for folder, taken by parent folder method

    glink, fid = get_shareable_link(file_path)

    link = 'https://drive.google.com/file/d/'+ getoutput("xattr -p 'user.drive.id' " + "'" + file_path + "'")   #for document
    # link = 'https://drive.google.com/drive/folders/'+ getoutput("xattr -p 'user.drive.id' " + "'" + file_path + "'")   #for folder

    fileNames.append(Path(files).stem)
    fileFullNames.append(Path(files).name)
    treeNos.append(Path(files).stem.split('_')[0])
    file_paths.append(file_path)
    links.append(link)
    fids.append(fid)

dic = {"fileName": fileNames, "fileFullName": fileFullNames, 'treeNo': treeNos, "Path": file_paths, "ShareLink": links, "fid": fids}
dfLink = pd.DataFrame(dic)
dfLink

/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/Screenshot 2025-08-28 at 1.46.11 PM.png
/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/Screenshot 2025-08-28 at 1.46.11 PM.png
1a3xxks07Hi2DygZi8OR8BrtS42qPN71Y
/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/T083_defect.png
/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/T083_defect.png
1F5gyB7uAaf8kdNW4qMCJB_ntXF2H5rLA
/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/T083_wshole.png
/content/drive/MyDrive/Muni/airTable/airTablePhotoUpload/T083_wshole.png
1ddcg2WZSAugVxycnjpLXCXnCG1G8pGje


,fileName,fileFullName,treeNo,Path,ShareLink,fid
0,Screenshot 2025-08-28 at 1.46.11 PM,Screenshot 2025-08-28 at 1.46.11 PM.png,Screenshot 2025-08-28 at 1.46.11 PM,/content/drive/MyDrive/Muni/airTable/airTableP...,https://drive.google.com/file/d/1a3xxks07Hi2Dy...,1a3xxks07Hi2DygZi8OR8BrtS42qPN71Y
1,T083_defect,T083_defect.png,T083,/content/drive/MyDrive/Muni/airTable/airTableP...,https://drive.google.com/file/d/1F5gyB7uAaf8kd...,1F5gyB7uAaf8kdNW4qMCJB_ntXF2H5rLA
2,T083_wshole,T083_wshole.png,T083,/content/drive/MyDrive/Muni/airTable/airTableP...,https://drive.google.com/file/d/1ddcg2WZSAugVx...,1ddcg2WZSAugVxycnjpLXCXnCG1G8pGje


# upload photos to airTable's attachment field

In [89]:
def uploadFiles(TreeNo, driveFid, fileFullName):

  if record['fields']['Tree No.'] == TreeNo:
    RECORD_ID = record['id']
    print(f'RECORD_ID: {RECORD_ID}') #reczx1Jy8hNHkjJ7r

    # not work
    # IMAGE_URL = 'https://drive.google.com/file/d/1ddcg2WZSAugVxycnjpLXCXnCG1G8pGje/view?usp=drive_link'

    # Convert Google Drive link to direct download link
    #drive_file_id = '1ddcg2WZSAugVxycnjpLXCXnCG1G8pGje'  # Extracted from your original link
    IMAGE_URL = f'https://drive.google.com/uc?id={driveFid}'

    r = requests.get(IMAGE_URL)
    if r.status_code == 200:
      # Save the image to a temporary file
      fileName = f'/content/{fileFullName}'

      with open(fileName, 'wb') as f:
          f.write(r.content)

      try:
        updated_record = table.upload_attachment(
            record_id = RECORD_ID,
            field = 'rawPhoto',  #import field 對應 airtable rawPhoto field
            filename = os.path.join('/content',fileFullName)
            # filename = IMAGE_URL
          )
      except Exception as e:
        print(f"Error uploading photo: {e}")


In [90]:
api = Api(token)
table = api.table(baseId,tableName)
records = table.all()

from pyairtable import Table

# Replace with your API key, base ID, and table name
#API_KEY = token
#BASE_ID = baseId
#TABLE_NAME = tableName

errorLst = []
# todo Tree No., drive_file_id, fileName
for x, record in enumerate(records):
  #print(record)
  # record: {'id': 'reczx1Jy8hNHkjJ7r', 'createdTime': '2025-08-28T07:55:27.000Z', 'fields': {'Tree No.': 'T083', 'Scientific Name': 'Lagerstroemia indica', 'Chinese Name': '紫薇', 'DBH (mm)': 188, 'Height (m)': 4, 'Crown Spread\n (m)': 5, 'Tree Form\n (Good/\n Fair/\n Poor)': 'Fair', 'Tree Health\n (Good/\n Fair/\n Poor)': 'Fair', 'Remark on Crown': ['Epicormics'], 'Remark on Trunk': ['Included bark', 'Multi-trunks'], 'loc': 'Baycrest', 'triage': 'null', 'gp': '1'}}
  if 'Tree No.' in record['fields'].keys():

    tree_no = record['fields']['Tree No.']

    #print(f'tree number: {tree_no}')
    if tree_no in dfLink['treeNo'].values:

      # find the indexs in dfLink
      indices = dfLink.index[dfLink['treeNo'] == tree_no].tolist()

      for i in indices:
        uploadFiles(dfLink['treeNo'][i], dfLink['fid'][i], dfLink['fileFullName'][i])
        print(f'upload: {dfLink['fid'][i]}')
  else:
    errorLst.append(x)
if len(errorLst)>0 :
  errorMsg = f'{len(errorLst)} files cannot upload'
else:
  errorMsg = ''
print(f'done. {errorMsg}')

RECORD_ID: reczx1Jy8hNHkjJ7r
upload: 1F5gyB7uAaf8kdNW4qMCJB_ntXF2H5rLA
RECORD_ID: reczx1Jy8hNHkjJ7r
upload: 1ddcg2WZSAugVxycnjpLXCXnCG1G8pGje
done. 1 files cannot upload
